# Documentación Técnica 

## __Función esp_trab.m__

### Propósito del algoritmo
Determinación y visualización geométrica del espacio de trabajo cartesiano (planos ortogonales XZ y XY, y volumen 3D) del robot __FANUC M10iD/8L__. El algoritmo opera aislando variables en el espacio articular e implementando la resolución iterativa de la cinemática directa.

### 1. Mapeo del plano de elevación (Vista XZ)

- Se definen 9 vectores de trayectoria (de traj_0 a traj_8) utilizando la función de interpolación polinómica jtraj.

- Cada segmento aísla cinemáticamente un grado de libertad específico (barridos individuales de la articulación 2 o 3) mientras mantiene el resto de la cadena cinemática estática en sus topes mecánicos. Esta restricción fuerza a la coordenada final a desplazarse estrictamente sobre la frontera teórica de máximo alcance.

- Las trayectorias se concatenan verticalmente mediante vertcat en la variable EDTXZ, estructurando una matriz de 900 filas por 6 columnas (900 vectores articulares de configuración espacial).

### 2. Resolución de cinemática directa y extracción cartesiana

- La cinemática directa permite conocer la localización del extremo del robot basándose exclusivamente en la posición de sus articulaciones.

- El bucle for itera sobre la matriz de configuración EDTXZ, ingresando cada vector $\overline{q}$ al método R.fkine(). Esta función ejecuta la multiplicación secuencial de las matrices de Denavit-Hartenberg, arrojando la matriz de transformación homogénea total ${}^{0}T_n$.   

- Para aislar la componente posicional, la función vectorial transl extrae únicamente el vector de traslación tridimensional $(x,y,z)$ de la matriz homogénea, almacenándolo en el arreglo Puntos_Lat. Posteriormente se grafican los vectores resultantes indexando las columnas 1 y 3 (ejes X y Z).


### 3. Mapeo del plano transversal (Vista XY)

- Se genera un único arreglo de interpolación (traj_planta) que barre la articulación de la base de $-170^\circ$ a $+170^\circ$.

- El manipulador se mantiene en máxima extensión radial. El mismo procesamiento iterativo con fkine y transl extrae las coordenadas de las columnas 1 y 2 para trazar la circunferencia de alcance transversal exterior.

### 4. Integración espacial combinada

- Se emplea la función plot3 para superponer simultáneamente los arreglos matriciales de las proyecciones Puntos_Lat y Puntos_Sup en un entorno euclídeo 3D.

- El comando R.teach() instancia el modelo cinemático interactivo de la estructura mecánica, permitiendo auditar visualmente que la cinemática directa computada coincida con los límites de la envolvente proyectada.

## __Función IK_numeric.m__

### Propósito del algoritmo
Diseñada para resolver la cinemática inversa de un robot manipulador utilizando métodos numéricos iterativos (basados en el Toolbox de Peter Corke). La función es capaz de calcular una trayectoria continua única o buscar múltiples configuraciones articulares válidas para una misma sucesión de poses cartesianas.

### 1. Entradas y salidas 

#### Entradas:
- R: Objeto del robot (SerialLink) que contiene los parámetros cinemáticos, físicos y límites articulares.
- T: Sucesión de poses cartesianas a alcanzar. Generalmente, una matriz de transformación homogénea o un arreglo de tamaño $4 \times 4 \times N$ (donde $N$ es la cantidad de puntos de la trayectoria generada con $jtraj$). 
- q_mejor: Booleano (true/false) que determina el modo de operación. Si es true, busca una única solución; si es false, busca múltiples soluciones alternativas, generando valores semilla aleatorios para las variables articulares. 
- q0: Vector de posiciones articulares semilla desde donde el algoritmo numérico comenzará a buscar.

#### Salidas:
- exito: Booleano que indica si el algoritmo logró encontrar al menos una solución válida.
- q_resultado: Contiene la trayectoria articular resultante. Si q_mejor = true, es una matriz de $N \times 6$. Si q_mejor = false, es un arreglo de celdas (cell array) donde cada celda contiene una matriz de trayectoria diferente.
- mensaje: Cadena de texto informativa que detalla el resultado de la ejecución o el motivo de la falla.

### 2. Parámetros de configuración interna
Al inicio del script, se inicializan las variables de salida por defecto asumiendo falla, lo cual es una buena práctica de programación segura. Luego, se definen tres parámetros clave para el comportamiento del algoritmo:
- __Máscara (M = [1,1,1,1,1,1]):__ Le indica a la función ikine que el robot tiene 6 grados de libertad activos y que debe calcular la posición y orientación en los ejes X, Y, Z, $\alpha$, $\beta$ y $\gamma$.
- __cantidad de semillas (seeds = 100):__ Define el límite de intentos que realizará el algoritmo para encontrar distintas semillas cuando se buscan múltiples soluciones.
- __Tolerancia (tol = 1e-2):__ Establece el margen de error máximo (en radianes) permitido para considerar si dos configuraciones articulares son idénticas o son verdaderamente soluciones distintas.

### 3. Modo de ejecución 1: única solución (q_mejor = true)
Este bloque se ejecuta cuando se requiere una solución rápida y directa.
- Utiliza un bloque de seguridad try...catch.
- Llama al solucionador numérico R.ikine pasándole directamente la semilla original q0 proporcionada por el usuario.
- Al pasar un conjunto de poses (T), ikine calcula la inversa del primer punto y automáticamente utiliza ese resultado como semilla del segundo punto, garantizando una trayectoria suave y continua.
- Si el algoritmo matemático diverge (falla), el bloque catch captura el error interno de MATLAB y lo devuelve en la variable mensaje sin colapsar el programa.

### 4. Modo de ejecución 2: múltiples soluciones (q_mejor = false)
Este es el núcleo avanzado de la función, diseñado para explorar el espacio articular y encontrar configuraciones mecánicas alternativas (ej. codo arriba vs. codo abajo) que logren la misma trayectoria cartesiana. Opera mediante un bucle principal regido por seeds.

__A. Generación aleatoria de semillas__
Para obligar al algoritmo numérico a encontrar soluciones distintas, el programa se sitúa en diferentes "puntos de partida". Se genera un nuevo vector q_semilla_nuevo donde cada articulación toma un valor aleatorio calculado estrictamente dentro de sus límites físicos (R.qlim(i,1) a R.qlim(i,2)).

__B. Escudo de iteración (isempty)__
Si la función ikine intenta resolver la trayectoria desde una semilla muy mala, puede fallar por completo y devolver un vector vacío. La instrucción if isempty(q_temp) intercepta esto y utiliza continue para abortar ese intento y probar inmediatamente con una nueva semilla aleatoria, evitando que el programa genere errores de índices.

__C. Validación mecánica (límites articulares)__
Dado que los métodos numéricos son abstractos, pueden generar trayectorias donde el robot atraviese sus propios topes físicos. El bloque es_valida extrae toda la columna de datos de cada articulación y compara sus valores mínimos y máximos contra R.qlim. Si durante cualquier punto de la trayectoria un eslabón excede su capacidad física, la solución se descarta.

__D. Filtro de soluciones repetidas__
Si una trayectoria es válida, se debe verificar que no sea idéntica a una ya guardada.
- Se compara únicamente la primera fila (el punto inicial) de la nueva trayectoria (vect_a) contra el punto inicial de las trayectorias ya almacenadas (vect_b).
- Se calcula la distancia euclidiana general usando la función norm(vect_a - vect_b).
- Si la distancia es menor a la tolerancia (tol), se clasifica como es_repetida = true y se descarta.
- Si es genuinamente nueva, se almacena en el arreglo de celdas soluciones_encontradas.

## Detalles importantes

1. Problemática de saltos de $2\pi$

Un ángulo es un punto sobre una circunferencia. Para representarlo como un número tenemos que "cortar" la circunferencia y desenrollarla en una recta. Habitualmente la recta queda en un intervalo $[-\pi,\pi]$.
Esto puede presentar un inconveniente, ya que tenemos articulaciones cuyos límites pueden caer fuera de este intervalo. 

En el siguiente fragmento de código del toolbox de Peter Corke podemos apreciar esta restricción del intervalo:

In [ ]:
% wrap angles for revolute joints
k = (q > pi) & revolutes;
q(k) = q(k) - 2*pi;

k = (q < -pi) & revolutes;
q(k) = q(k) + 2*pi;

nm = norm(W*e);

Esto hace que el script "envuelva" todas las soluciones en el intervalo $[-\pi,\pi]$. 
Por ejemplo, si la articulación está barriendo un ángulo desde 179° hasta 181° el script va a interpretar lo siguiente:

- 179° -> dentro del intervalo
- 180° -> en el límite del intervalo
- 181° -> FUERA del intervalo -> corrección -> -179° (ahora dentro del intervalo).

Esto presenta un problema grave, ya que produjo un salto de 180° a -179° (prácticamente $2\pi$). A nivel matemático no es un problema, ya que ambos ángulos representan el mismo valor numéricamente hablando ($q = q+2\pi k$, con k entero positivo), pero físicamente implica que la articulación de una vuelta completa en un instante, eso puede provocar un desastre. 

Para solucionar esto hay que sumarle a la actual una cantidad entera de vueltas (múltiplos de 360°) para dejarla lo más cerca posible de la anterior:

In [ ]:
for k = 2:size(q_temp,1)
    n = round((q_temp(k,:) - q_temp(k-1,:)) / (2*pi));
    q_temp(k,:) = q_temp(k,:) - 2*pi*n;
end

Esto compara la solución nueva (vector con las 6 articulaciones) con la muestra anterior. La linea dentro del for se pregunta cuantas vueltas completas hay entre lo que queremos y lo que tenemos. Esta diferencia se divide por $2\pi$ y se redondea al entero más cercano. 

In [ ]:
q_ref = 179°, q = -179°
(179 - (-179)) / 360 = 358 / 360 = 0.994  ->  round -> k = 1
qi = -179 + 360*1 = 181°

Para el caso en el cual no hay un salto, la cuenta da casi cero:

In [ ]:
q_ref = 10°, q = 12°
(10 - 12) / 360 = -0.0056  ->  round -> k = 0
qi = 12°   (no se toca)

2. Verificación de la pose del robot

Después de cada ikine se hace uso de la cinemática directa para ver donde llevaría al robot ese valor de q.